# Notebook 01 — Data, weak labels, splits & graphs

**What this notebook does (one major step):**
1. Load PROTAC-DB 4.0 Excel files
2. Build warhead / E3 reference libraries
3. Weak-label each PROTAC (dictionary match)
4. Keep only labels that **reassemble** into 3 valid fragments
5. Build evaluation splits (random, unseen warhead/E3, fingerprint-OOD, newer-ID)
6. Convert molecules to graph tensors and save everything under `data/processed/`

No external `src` package — all helper code is in the cells below.


## 0. Paths

In [ ]:
# Project root = parent of notebooks/ (or cwd if already in Yashi/)
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "Protac_4.0_database_files_downloaded"
PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "outputs"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)


## 1. Helper functions — labelling & reassembly

In [ ]:
from __future__ import annotations

import io
import math
import random
from collections import Counter, deque
from dataclasses import dataclass
from typing import Iterable, Optional

import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Draw, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D

RDLogger.DisableLog("rdApp.*")

WARHEAD, LINKER, E3 = 0, 1, 2
CLASS_NAMES = {WARHEAD: "warhead", LINKER: "linker", E3: "e3_ligand"}
CLASS_COLORS = {
    WARHEAD: (1.0, 0.35, 0.35),   # pink
    LINKER:  (0.30, 0.55, 1.0),   # blue
    E3:      (0.30, 0.85, 0.35),  # green
}

# =============================================================================
# 1. Weak labelling  (dictionary + reassembly filter)
# =============================================================================

@dataclass
class RefFragment:
    smiles: str
    mol: Chem.Mol
    n_atoms: int


def prepare_reference_library(
    smiles_list: Iterable[str], min_atoms: int = 5, max_atoms: int = 60
) -> list[RefFragment]:
    """Parse + filter reference SMILES; sort largest-first for greedy match."""
    refs: list[RefFragment] = []
    for smi in smiles_list:
        if not isinstance(smi, str) or not smi:
            continue
        m = Chem.MolFromSmiles(smi)
        if m is None:
            continue
        n = m.GetNumHeavyAtoms()
        if not (min_atoms <= n <= max_atoms):
            continue
        refs.append(RefFragment(Chem.MolToSmiles(m), m, n))
    refs.sort(key=lambda r: r.n_atoms, reverse=True)
    return refs


def _best_match(mol: Chem.Mol, refs: list[RefFragment],
                forbidden: Optional[set[int]] = None) -> Optional[tuple[set[int], str]]:
    """Return (atoms_of_best_match, ref_smiles) or None."""
    forbidden = forbidden or set()
    n_target = mol.GetNumAtoms()
    for ref in refs:
        if ref.n_atoms > n_target:
            continue
        matches = mol.GetSubstructMatches(ref.mol, uniquify=True, useChirality=False)
        for match in matches:
            atoms = set(match)
            if atoms.isdisjoint(forbidden):
                return atoms, ref.smiles
    return None


def _boundary_bonds(mol: Chem.Mol, labels: list[int]) -> list[int]:
    return [b.GetIdx() for b in mol.GetBonds()
            if labels[b.GetBeginAtomIdx()] != labels[b.GetEndAtomIdx()]]


def reassembles(mol: Chem.Mol, labels: list[int]) -> tuple[bool, int]:
    """Cut molecule at boundary bonds; return (success, n_fragments).

    Reassembly succeeds when cutting produces exactly 3 chemically valid
    fragments and no atoms are lost.  This is the Ribes-style validation
    of a splitting, applied to *any* atom-level labelling.
    """
    b_idx = _boundary_bonds(mol, labels)
    if not b_idx:
        return False, 1
    frag_mol = Chem.FragmentOnBonds(mol, b_idx, addDummies=True)
    frags = Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False)
    if len(frags) != 3:
        return False, len(frags)
    atoms_seen = 0
    for f in frags:
        try:
            Chem.SanitizeMol(f)
        except Exception:
            return False, len(frags)
        atoms_seen += sum(1 for a in f.GetAtoms() if a.GetAtomicNum() != 0)
    return (atoms_seen == mol.GetNumAtoms()), len(frags)


def label_protac(smiles: str, wh_refs: list[RefFragment], e3_refs: list[RefFragment],
                 require_reassembly: bool = True) -> Optional[dict]:
    """Weak-label one PROTAC. Return dict or None on failure."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    n = mol.GetNumAtoms()
    if not (15 <= n <= 120):
        return None
    wh = _best_match(mol, wh_refs)
    if wh is None:
        return None
    e3 = _best_match(mol, e3_refs, forbidden=wh[0])
    if e3 is None:
        return None
    labels = [LINKER] * n
    for a in wh[0]:
        labels[a] = WARHEAD
    for a in e3[0]:
        labels[a] = E3
    if LINKER not in labels:
        return None
    if require_reassembly:
        ok, _ = reassembles(mol, labels)
        if not ok:
            return None
    return {
        "smiles": Chem.MolToSmiles(mol),
        "labels": labels,
        "wh_ref": wh[1],
        "e3_ref": e3[1],
        "n_warhead": len(wh[0]),
        "n_linker": labels.count(LINKER),
        "n_e3": len(e3[0]),
    }


## 2. Helper functions — train/val/test splits

In [ ]:
# =============================================================================
# 2. Splits  (random + unseen ligand + true OOD)
# =============================================================================

def _morgan_bits(smiles: str, n_bits: int = 2048, radius: int = 2) -> np.ndarray:
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros(n_bits, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def _tanimoto(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / union) if union else 0.0


def make_splits(records: list[dict], seed: int = 42) -> dict[str, dict[str, list[int]]]:
    """Build IID + chemotype-held-out + fingerprint-OOD + newer-ID splits."""
    n = len(records)
    idx = list(range(n))
    rng = random.Random(seed)

    # ---- random 80/10/10
    rng.shuffle(idx)
    n_tr = int(0.8 * n); n_val = int(0.1 * n)
    rand = {"train": idx[:n_tr], "val": idx[n_tr:n_tr + n_val], "test": idx[n_tr + n_val:]}

    # ---- unseen-warhead / unseen-e3: rarest refs held out
    def by_ref(key):
        groups: dict[str, list[int]] = {}
        for i, r in enumerate(records):
            groups.setdefault(r[key], []).append(i)
        ordered = sorted(groups.items(), key=lambda kv: len(kv[1]))  # rarest first
        val_ids, test_ids = [], []
        for ref, ids in ordered:
            if len(val_ids) < n * 0.1:
                val_ids.extend(ids)
            elif len(test_ids) < n * 0.1:
                test_ids.extend(ids)
            else:
                break
        val_set, test_set = set(val_ids), set(test_ids)
        train_ids = [i for i in range(n) if i not in val_set and i not in test_set]
        return {"train": train_ids, "val": val_ids, "test": test_ids}

    # ---- fingerprint-dissimilar OOD: molecules least similar to a random pool
    # This approximates "different source / novel chemotype" without needing an
    # external database: test molecules have low max-Tanimoto to the train pool.
    fps = [_morgan_bits(r["smiles"]) for r in records]
    pool = list(range(n))
    rng.shuffle(pool)
    pool_train = pool[: int(0.7 * n)]
    remaining = pool[int(0.7 * n):]
    # score remaining by max Tanimoto to pool_train
    scores = []
    for i in remaining:
        mx = max(_tanimoto(fps[i], fps[j]) for j in pool_train[: min(200, len(pool_train))])
        scores.append((mx, i))
    scores.sort()  # most dissimilar first
    n_test = max(1, int(0.1 * n)); n_val_fp = max(1, int(0.1 * n))
    fp_test = [i for _, i in scores[:n_test]]
    fp_val = [i for _, i in scores[n_test:n_test + n_val_fp]]
    fp_held = set(fp_test) | set(fp_val)
    fp_train = [i for i in range(n) if i not in fp_held]
    fp_ood = {"train": fp_train, "val": fp_val, "test": fp_test}

    # ---- newer-ID split: high Compound IDs act as "newer chemotypes"
    # (PROTAC-DB assigns IDs roughly chronologically). Falls back to random
    # order if compound_id is missing.
    def _cid(r):
        v = r.get("compound_id")
        try:
            return int(v)
        except Exception:
            return hash(r["smiles"]) % 10_000_000

    ordered_by_id = sorted(range(n), key=lambda i: _cid(records[i]))
    n_new = max(1, int(0.1 * n)); n_val_new = max(1, int(0.1 * n))
    new_test = ordered_by_id[-n_new:]
    new_val = ordered_by_id[-(n_new + n_val_new):-n_new]
    new_held = set(new_test) | set(new_val)
    new_train = [i for i in ordered_by_id if i not in new_held]
    newer = {"train": new_train, "val": new_val, "test": new_test}

    return {
        "random":              rand,
        "unseen_warhead":      by_ref("wh_ref"),
        "unseen_e3":           by_ref("e3_ref"),
        "fingerprint_ood":     fp_ood,
        "newer_chemotype":     newer,
    }


## 3. Helper functions — molecule → graph tensors

In [ ]:
# =============================================================================
# 3. Graph construction  (no PyTorch Geometric)
# =============================================================================

ATOM_TYPES = ["C", "N", "O", "S", "F", "Cl", "Br", "I", "P", "B", "Si", "Se", "Other"]
HYBRIDS = [Chem.HybridizationType.SP, Chem.HybridizationType.SP2,
           Chem.HybridizationType.SP3, Chem.HybridizationType.SP3D,
           Chem.HybridizationType.SP3D2]
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE,
              Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]


def _oh(v, choices):
    x = [0.0] * (len(choices) + 1)
    try:
        x[choices.index(v)] = 1.0
    except ValueError:
        x[-1] = 1.0
    return x


def atom_features(a: Chem.Atom) -> list[float]:
    return (_oh(a.GetSymbol(), ATOM_TYPES) + _oh(a.GetHybridization(), HYBRIDS)
            + [float(a.GetDegree()), float(a.GetFormalCharge()),
               float(a.GetTotalNumHs()), float(a.GetIsAromatic()),
               float(a.IsInRing())])


def bond_features(b: Chem.Bond) -> list[float]:
    return _oh(b.GetBondType(), BOND_TYPES) + [float(b.GetIsConjugated()),
                                               float(b.IsInRing())]


ATOM_FEATURE_DIM = len(atom_features(Chem.MolFromSmiles("C").GetAtomWithIdx(0)))
BOND_FEATURE_DIM = len(bond_features(Chem.MolFromSmiles("CC").GetBondWithIdx(0)))


def smiles_to_graph(smiles: str, labels: Optional[list[int]] = None) -> Optional[dict]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    n = mol.GetNumAtoms()
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float32)
    src, dst, eattr = [], [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        f = bond_features(b)
        src += [i, j]; dst += [j, i]; eattr += [f, f]
    ei = torch.tensor([src, dst], dtype=torch.long) if src else torch.zeros(2, 0, dtype=torch.long)
    ea = torch.tensor(eattr, dtype=torch.float32) if eattr else torch.zeros(0, BOND_FEATURE_DIM)
    g = {"x": x, "edge_index": ei, "edge_attr": ea, "n_atoms": n, "smiles": smiles}
    if labels is not None:
        if len(labels) != n:
            return None
        g["y"] = torch.tensor(labels, dtype=torch.long)
    return g


def collate(graphs: list[dict]) -> dict:
    """Block-diagonal batch. `batch` maps each atom to its molecule id."""
    xs, eis, eas, ys, batch = [], [], [], [], []
    off = 0
    for i, g in enumerate(graphs):
        xs.append(g["x"]); eis.append(g["edge_index"] + off); eas.append(g["edge_attr"])
        batch.append(torch.full((g["n_atoms"],), i, dtype=torch.long))
        if "y" in g:
            ys.append(g["y"])
        off += g["n_atoms"]
    out = {
        "x": torch.cat(xs, 0),
        "edge_index": torch.cat(eis, 1),
        "edge_attr": torch.cat(eas, 0),
        "batch": torch.cat(batch, 0),
        "n_graphs": len(graphs),
    }
    if ys:
        out["y"] = torch.cat(ys, 0)
    return out


## 4. Load Excel tables

In [ ]:
import pandas as pd
from rdkit import Chem, RDLogger
from tqdm import tqdm
RDLogger.DisableLog("rdApp.*")

protac_df  = pd.read_excel(DATA_DIR / "protac.xlsx")[["Compound ID", "Target", "E3 ligase", "Smiles"]].dropna(subset=["Smiles"])
warhead_df = pd.read_excel(DATA_DIR / "warhead.xlsx")[["Compound ID", "Target", "Smiles"]].dropna(subset=["Smiles"])
e3_df      = pd.read_excel(DATA_DIR / "e3_ligand.xlsx")[["Compound ID", "Target", "Smiles"]].dropna(subset=["Smiles"])
print("raw counts — protac:", len(protac_df), "warhead:", len(warhead_df), "e3:", len(e3_df))


## 5. Filter parseable PROTACs (15–120 atoms) + Compound-ID map

In [ ]:
def parseable(s):
    m = Chem.MolFromSmiles(s)
    return m is not None and 15 <= m.GetNumAtoms() <= 120

protac_df = protac_df[protac_df["Smiles"].apply(parseable)].reset_index(drop=True)

smi_to_cid = {}
for _, row in protac_df.iterrows():
    m = Chem.MolFromSmiles(row["Smiles"])
    if m is None:
        continue
    cid = int(row["Compound ID"]) if pd.notna(row["Compound ID"]) else -1
    smi_to_cid[Chem.MolToSmiles(m)] = cid

print("parseable PROTACs:", len(protac_df))


## 6. Build reference libraries (largest fragments first)

In [ ]:
wh_refs = prepare_reference_library(warhead_df["Smiles"].tolist())
e3_refs = prepare_reference_library(e3_df["Smiles"].tolist())
print("warhead refs:", len(wh_refs), "  e3 refs:", len(e3_refs))


## 7. Weak-label PROTACs (dictionary + reassembly filter)

In [ ]:
import json, random
from tqdm import tqdm

SAMPLE = 6000   # set None to use full DB (slower)
random.seed(0)
smiles_list = protac_df["Smiles"].tolist()
if SAMPLE is not None:
    smiles_list = random.sample(smiles_list, min(SAMPLE, len(smiles_list)))

print("labeling", len(smiles_list), "PROTACs...")
records = []
dropped_by_reasm = 0
for s in tqdm(smiles_list):
    rec = label_protac(s, wh_refs, e3_refs, require_reassembly=True)
    if rec is not None:
        rec["compound_id"] = smi_to_cid.get(rec["smiles"], -1)
        records.append(rec)
    else:
        if label_protac(s, wh_refs, e3_refs, require_reassembly=False) is not None:
            dropped_by_reasm += 1

print(f"kept: {len(records)}   dropped by reassembly filter: {dropped_by_reasm}")

with open(PROCESSED / "labeled_protacs.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")
print("saved", PROCESSED / "labeled_protacs.jsonl")


## 8. Build five evaluation splits

In [ ]:
import pickle
print("building splits (fingerprint OOD may take a minute)...")
splits = make_splits(records, seed=42)
for name, s in splits.items():
    print(f"  {name:18s}  train={len(s['train']):4d}  val={len(s['val']):4d}  test={len(s['test']):4d}")

with open(PROCESSED / "splits.pkl", "wb") as f:
    pickle.dump(splits, f)

with open(PROCESSED / "refs.pkl", "wb") as f:
    pickle.dump({"warheads": [r.smiles for r in wh_refs],
                 "e3": [r.smiles for r in e3_refs]}, f)
print("saved splits.pkl and refs.pkl")


## 9. Convert to graphs and save tensors

In [ ]:
import torch
from tqdm import tqdm

graphs = []
for r in tqdm(records):
    graphs.append(smiles_to_graph(r["smiles"], r["labels"]))

torch.save({"graphs": graphs, "atom_dim": ATOM_FEATURE_DIM}, PROCESSED / "graphs.pt")
print("graphs:", len(graphs), "atom_dim:", ATOM_FEATURE_DIM)
print("DONE — Notebook 01 complete.")
